# Download and refresh local price CSV files

Uses `download_adj_close_prices` from `data_loader.py` with provider fallback chain: `yfinance` -> `defeatbeta_api`.

It downloads each ticker in a specified date range and saves one CSV per ticker in `data/` with the same format used by the project (`Date,Adj Close`).

In [10]:
from __future__ import annotations

from pathlib import Path
import importlib
import time
import pandas as pd

import data_loader as _data_loader
importlib.reload(_data_loader)
download_adj_close_prices = _data_loader.download_adj_close_prices

In [11]:
TICKERS = [
    "CSSPX.MI",
    "AVUV",
    "VTI",
    "VBR",
    "IEF",
    "VUTY.AS",
    "SWDA.MI",
    "EIMI.MI",
    "CSNDX.MI",
    "IUSQ.MI",
    "VWCE.MI",
    "EXS1.MI",
    "MEUD.MI",
    "SMEA.MI",
    "XD9U.MI",
    "XMME.MI",
    "CSSX5E.MI",
    "EMUL.MI",
    "XLRE",
    "GOVT",
    "TLH",
    "LTPZ",
    "XTLT.TO",
    "UTHY",
    "TIP",
    "GLD",
    "IGLA.L",
    "XG7S.DE",
    "SEGA.MI",
    "VAGF.DE",
    "BTC=F",
    "SI=F",
    "CL=F",
    "CSH.PA",
    "PDBC",
    "VGLT",
    "VGSH",
    "VWRA.L",
]

In [12]:
# Simple updater focused on VTI and VBR only
TICKERS = ["VTI", "VBR"]

# Used only if a local file does not exist yet
FALLBACK_START_DATE = "2010-04-03"
END_DATE = "2026-04-20"

# Small delay between requests helps reduce provider throttling
COOLDOWN_SECONDS = 4

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tickers: {TICKERS}")
print(f"Fallback start: {FALLBACK_START_DATE}")
print(f"End date: {END_DATE}")
print(f"Output dir: {DATA_DIR.resolve()}")

Tickers: ['VTI', 'VBR']
Fallback start: 2010-04-03
End date: 2026-04-20
Output dir: /Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data


In [13]:
results = []

for i, ticker in enumerate(TICKERS, start=1):
    print(f"[{i}/{len(TICKERS)}] Updating {ticker}...")
    status = "ok"
    rows = 0
    downloaded_rows = 0
    error = ""

    out_path = DATA_DIR / f"{ticker}.csv"
    req_end = pd.Timestamp(END_DATE)
    one_day = pd.Timedelta(days=1)

    try:
        existing_series = None
        start_ts = pd.Timestamp(FALLBACK_START_DATE)

        if out_path.exists():
            existing_df = pd.read_csv(out_path, parse_dates=["Date"])
            if not existing_df.empty and {"Date", "Adj Close"}.issubset(existing_df.columns):
                existing_series = (
                    existing_df.set_index("Date")["Adj Close"]
                    .astype(float)
                    .dropna()
                    .sort_index()
                )
                if not existing_series.empty:
                    start_ts = existing_series.index.max() + one_day

        if start_ts > req_end:
            print("    already up to date; no download needed")
            final_series = existing_series
            if final_series is None or final_series.empty:
                raise RuntimeError("No local data available and no date range to download.")
        else:
            new_prices = download_adj_close_prices(
                tickers=[ticker],
                start=start_ts.strftime("%Y-%m-%d"),
                end=req_end.strftime("%Y-%m-%d"),
            )
            new_series = new_prices[ticker].dropna().sort_index()
            downloaded_rows = int(new_series.shape[0])

            if existing_series is not None and not existing_series.empty:
                final_series = pd.concat([existing_series, new_series]).sort_index()
                final_series = final_series[~final_series.index.duplicated(keep="last")]
            else:
                final_series = new_series

        out_df = final_series.to_frame(name="Adj Close")
        out_df.index.name = "Date"
        out_df.to_csv(out_path, date_format="%Y-%m-%d")

        rows = int(out_df.shape[0])
        print(f"    downloaded {downloaded_rows} new rows; total rows now {rows}")
    except Exception as exc:
        status = "failed"
        error = str(exc)
        print(f"    FAILED: {ticker}: {error}")

    results.append(
        {
            "ticker": ticker,
            "status": status,
            "downloaded_rows": downloaded_rows,
            "total_rows": rows,
            "error": error,
        }
    )

    if i < len(TICKERS):
        time.sleep(COOLDOWN_SECONDS)

summary = pd.DataFrame(results)
summary

[1/2] Updating VTI...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:324: RuntimeWarning: Local cache miss for symbols: ['VTI']. Attempting online providers.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/yfinance/scrapers/history.py:169: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()

1 Failed download:
['VTI']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:377: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['VTI']
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:386: RuntimeWarning: Yahoo Finance batch response returned no usable rows for all uncached symbols. Likely temporary provider throttling; retrying per ticker with backoff.
  war

KeyboardInterrupt: 